# Data Exploration

Runs the data loading and preprocessing pipeline step by step so each stage can be checked in memory.

## Imports and Paths

In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# Make the repo root importable from notebooks/
sys.path.insert(0, os.path.abspath(".."))

from data import (
    TRAIN_SPLITS,
    VAL_SPLITS,
    TEST_SPLITS,
    load_view,
    merge_labels,
    split_by_fold,
    clean_frames,
    get_common_patients,
    build_mask_dict,
    preprocess_view,
    EchoDataset,
    compute_ef_stats,
)

In [ ]:
#-------------------
# Path to dataset: the folder containing the A4C/ and PSAX/ folders
DATASET_ROOT = "path/to/dataset"

# Path to save the extracted frames (.npy) to
PROCESSED_ROOT = "path/to/processed"

a4c_dir  = os.path.join(DATASET_ROOT, "A4C")
psax_dir = os.path.join(DATASET_ROOT, "PSAX")

processed_a4c  = os.path.join(PROCESSED_ROOT, "a4c")
processed_psax = os.path.join(PROCESSED_ROOT, "psax")

Checks if data loaded in correctly

In [ ]:
print(os.listdir(a4c_dir))
print(os.listdir(psax_dir))

## Data loading and cleanup

A4C and PSAX Data

In [ ]:
# Grab the csv list and volume tracings for a4c and psax
df_a4c, df_a4c_volume = load_view(a4c_dir)
df_psax, df_psax_volume = load_view(psax_dir)

# Print the lengths to ensure they were grabbed
print(len(df_a4c))
print(len(df_a4c_volume))
print(len(df_psax))
print(len(df_psax_volume))

Check for Unique Patients

In [ ]:
# Check if the patient_id column correctly captures the patient IDs
print(f"A4C DF: {df_a4c['patient_id'].nunique()}")
print(f"A4C Volume DF: {df_a4c_volume['patient_id'].nunique()}")

print(f"PSAX DF: {df_psax['patient_id'].nunique()}")
print(f"PSAX Volume DF: {df_psax_volume['patient_id'].nunique()}")

# Check the amount of patients
a4c_patients = set(df_a4c["patient_id"])
psax_patients = set(df_psax["patient_id"])

print(f"A4C DF SET: {len(a4c_patients)}")
print(f"PSAX DF SET: {len(psax_patients)}")

only_a4c = a4c_patients - psax_patients
print(f"Only A4C patients: {len(only_a4c)}")

only_psax = psax_patients - a4c_patients
print(f"Only PSAX patients:{len(only_psax)}")

### Train/Val/Test Split

In [ ]:
# Split the a4c data
train_a4c = split_by_fold(df_a4c, TRAIN_SPLITS)
val_a4c   = split_by_fold(df_a4c, VAL_SPLITS)
test_a4c  = split_by_fold(df_a4c, TEST_SPLITS)

# Split the psax data
train_psax = split_by_fold(df_psax, TRAIN_SPLITS)
val_psax   = split_by_fold(df_psax, VAL_SPLITS)
test_psax  = split_by_fold(df_psax, TEST_SPLITS)

print(len(train_a4c), len(val_a4c), len(test_a4c))
print(len(train_psax), len(val_psax), len(test_psax))

Ensure there are no intersections between train/val/test splits

In [ ]:
# Outputs how many patients are in each split
# Used to ensure all patients are accounted for
print(train_a4c["patient_id"].nunique(), val_a4c["patient_id"].nunique(), test_a4c["patient_id"].nunique())
print(train_psax["patient_id"].nunique(), val_psax["patient_id"].nunique(), test_psax["patient_id"].nunique())

all_train = set(train_a4c["patient_id"]).union(train_psax["patient_id"])
all_val = set(val_a4c["patient_id"]).union(val_psax["patient_id"])
all_test = set(test_a4c["patient_id"]).union(test_psax["patient_id"])

# Test for patient disjoint
print(len(all_train & all_val))
print(len(all_train & all_test))
print(len(all_val & all_test))

Merge CSV

In [ ]:
df_a4c_volume = merge_labels(df_a4c_volume, df_a4c)
df_psax_volume = merge_labels(df_psax_volume, df_psax)

Set up Volume split

In [ ]:
train_a4c_volume = split_by_fold(df_a4c_volume, TRAIN_SPLITS)
val_a4c_volume   = split_by_fold(df_a4c_volume, VAL_SPLITS)
test_a4c_volume  = split_by_fold(df_a4c_volume, TEST_SPLITS)

train_psax_volume = split_by_fold(df_psax_volume, TRAIN_SPLITS)
val_psax_volume   = split_by_fold(df_psax_volume, VAL_SPLITS)
test_psax_volume  = split_by_fold(df_psax_volume, TEST_SPLITS)

print(df_a4c_volume.columns)
print(df_psax_volume.columns)

print(df_a4c_volume.head())
print(df_psax_volume.head())

In [ ]:
a4c_train_ids = set(train_a4c_volume["patient_id"].unique())
psax_train_ids = set(train_psax_volume["patient_id"].unique())

print("A4C count:", len(a4c_train_ids))
print("PSAX count:", len(psax_train_ids))
print("Intersection:", len(a4c_train_ids & psax_train_ids))
print("A4C only:", len(a4c_train_ids - psax_train_ids))
print("PSAX only:", len(psax_train_ids - a4c_train_ids))

# Patients with both an A4C and PSAX video in each split
train_patients = get_common_patients(train_a4c_volume, train_psax_volume)
val_patients   = get_common_patients(val_a4c_volume, val_psax_volume)
test_patients  = get_common_patients(test_a4c_volume, test_psax_volume)

print("\n=================================================")
print("Common patients:", len(train_patients))
print("Common patients:", len(val_patients))
print("Common patients:", len(test_patients))

common_patients_all = train_patients | val_patients | test_patients
print("Total common patients:", len(common_patients_all))

## Dataset Preprocessing

In [ ]:
# Filters out any invalid frames and converts them to 0-based indices
df_a4c_volume = clean_frames(df_a4c_volume)
df_psax_volume = clean_frames(df_psax_volume)

# Create the ground truth segmentation masks
mask_dict_a4c = build_mask_dict(df_a4c_volume)
mask_dict_psax = build_mask_dict(df_psax_volume)

print(len(mask_dict_a4c))
print(len(mask_dict_psax))

Frame extraction: saves every video of the common patients as a .npy file

In [ ]:
preprocess_view(df_a4c, os.path.join(a4c_dir, "Videos"), processed_a4c, common_patients_all)
preprocess_view(df_psax, os.path.join(psax_dir, "Videos"), processed_psax, common_patients_all)

## Dataset Setup

In [ ]:
# Ensure no patient is in more than one set
overlap = len(train_patients & val_patients & test_patients)
if overlap == 0:
    print("No Overlap")
else:
    print("WARNING PATIENT OVERLAP DETECTED")

train_dataset = EchoDataset(train_a4c_volume, train_psax_volume, processed_a4c, processed_psax, train_patients, mask_dict_a4c, mask_dict_psax)
val_dataset   = EchoDataset(val_a4c_volume, val_psax_volume, processed_a4c, processed_psax, val_patients, mask_dict_a4c, mask_dict_psax)
test_dataset  = EchoDataset(test_a4c_volume, test_psax_volume, processed_a4c, processed_psax, test_patients, mask_dict_a4c, mask_dict_psax)

EF normalization stats from the train set.

The original training run gave EF_MEAN: 60.9132, EF_STD: 10.5347

In [ ]:
EF_MEAN, EF_STD = compute_ef_stats(train_dataset)

print("EF_MEAN:", EF_MEAN)
print("EF_STD:", EF_STD)

Dataset Sanity Check:

This cell is used only to ensure the data preprocessing has worked as intended.

The intended output should be:

- Train Dataset Pairs: 2609
- Validation Dataset Pairs: 343
- Test Dataset Pairs: 374

The rest of the results will show 3 samples:
- A4C ED and ES Frames
- PSAX ED and ES Frames

In [ ]:
print(f"Train Dataset Pairs: {len(train_dataset)}")
print(f"Validation Dataset Pairs: {len(val_dataset)}")
print(f"Test Dataset Pairs: {len(test_dataset)}")
print("==============================\n")


def show(img, mask, title):
    plt.figure()
    plt.imshow(img[0], cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()


dataset = train_dataset
for i in range(3):

    sample = dataset[i]

    a4c = sample["a4c"]
    psax = sample["psax"]

    print("\n==============================")
    print("Sample:", i)
    print("================================\n")

    ed = sample["ed_a4c"]
    es = sample["es_a4c"]

    ed_p = sample["ed_psax"]
    es_p = sample["es_psax"]

    show(a4c[ed], sample["seg_a4c"][ed], "A4C ED overlay")
    show(a4c[es], sample["seg_a4c"][es], "A4C ES overlay")

    show(psax[ed_p], sample["seg_psax"][ed_p], "PSAX ED overlay")
    show(psax[es_p], sample["seg_psax"][es_p], "PSAX ES overlay")